# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
# Print basic info
print(f"{meta.name}: {meta.description}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")
print(f"Published: {getattr(meta, 'datePublished', 'N/A')}")
print(f"License: {getattr(meta, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

First, enumerate all record sets and their fields using their `@id`s.

In [ ]:
# List all available record sets, their @id's and fields
record_sets = list(dataset.record_sets())

print(f"Found {len(record_sets)} record set(s) in the dataset.")
for rs in record_sets:
    print(f"Record Set: {rs['@id']} - {rs.get('name', '')}")
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for f in fields:
            fid = f['@id'] if isinstance(f, dict) else f
            print(f"    - {fid}")
    else:
        print("  No fields defined.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In the FAIR^2 dataset, the main data record set typically has `@id` similar to `cr:RecordSet_1` (use exact value from above cell).

In [ ]:
# Pick all record set @id's for extraction
record_set_ids = [rs['@id'] for rs in record_sets]
dfs = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dfs[record_set_id] = df
            print(f"Loaded {len(df)} records for record set {record_set_id}.")
            print(f"Columns: {df.columns.tolist()}")
        else:
            print(f"No records loaded for record set {record_set_id}.")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")
        continue
# For further analysis, pick the first dataframe with data
main_record_set_id = next((k for k, v in dfs.items() if not v.empty), None)
if main_record_set_id:
    print(f"\nMain record set selected: {main_record_set_id}")
    print(dfs[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, and grouping by attributes.

Choose a numeric field (e.g., `cr:field_normalized_abundance`) and a group field (e.g., `cr:field_sample`).

In [ ]:
# List columns for main dataframe
if main_record_set_id:
    df = dfs[main_record_set_id]
    print(f"Available fields: {df.columns.tolist()}")
    # Heuristically pick a numeric field and a group field
    numeric_candidates = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or 'abundance' in col.lower() or 'count' in col.lower()]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        print("No obvious numeric field found. Picking the first column.")
        numeric_field = df.columns[0]
    group_candidates = [col for col in df.columns if 'sample' in col.lower() or 'group' in col.lower() or 'class' in col.lower()]
    if group_candidates:
        group_field = group_candidates[0]
    else:
        group_field = None
    print(f"Numeric field candidate: {numeric_field}")
    print(f"Group field candidate: {group_field}")

    # Try to convert the numeric field to float, if possible
    with pd.option_context('mode.use_inf_as_na', True):
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Filter records where value > threshold (10 for demonstration)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}, found {len(filtered_df)} rows.")
    print(filtered_df.head())

    # Add normalized column
    if len(filtered_df) > 0:
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by an attribute, e.g., a 'sample' or 'group' column
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and len(filtered_df) > 0:
    # Histogram of the normalized numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of normalized {numeric_field}")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group, if available
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"Distribution of {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The dataset contains information on protein abundance and other properties from mass spectrometry analysis of extracellular vesicles from human mast cells.
* Data can be explored and grouped by experimental or biosample labels.
* Further analysis can be performed, such as statistical comparison between groups or protein classes, or deeper investigation into hypotheses presented by the dataset authors.

The FAIR<sup>2</sup> Croissant schema and the `mlcroissant` library allow for easily reproducible, machine-actionable data workflows.